In [ ]:
import os
import torch

class Config:
    # --- Paths ---
    PRETRAINED_MODEL_PATH = "/content/drive/MyDrive/path_to_your_checkpoint_folder" 
    PRODUCT_DATA_PATH = "preprocessed_products.parquet"
    QUERY_DATA_PATH = "preprocessed_queries.parquet"
    
    # --- Data Splitting ---
    TEST_SIZE = 0.1
    VAL_SIZE = 0.1
    SEED = 42
    
    # --- Negative Sampling Ratios ---
    NEG_TYPE_1_RATIO = 0.4  # Impression but not clicked
    NEG_TYPE_2_RATIO = 0.3  # Random same category
    NEG_TYPE_3_RATIO = 0.3  # Random other category
    
    # --- Model Architecture ---
    MAX_LENGTH = 64
    PROJECTION_DIM = 128
    DROPOUT = 0.1
    
    # --- Training ---
    BATCH_SIZE = 32
    VAL_BATCH_SIZE = 64 # Larger batch size for validation since no backprop
    NUM_EPOCHS = 5
    LEARNING_RATE = 2e-5
    WEIGHT_DECAY = 0.01
    TEMPERATURE = 0.07
    LOG_STEPS = 50          # Record training loss every N steps
    
    # --- Loss Weights ---
    LAMBDA_CLS = 0.1
    
    # --- Device ---
    DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

print(f"Running on device: {Config.DEVICE}")

In [ ]:
import pandas as pd
import numpy as np
import ast
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

def load_and_prep_data():
    print("Loading data...")
    df_products = pd.read_parquet(Config.PRODUCT_DATA_PATH)
    df_queries = pd.read_parquet(Config.QUERY_DATA_PATH)
    
    # --- 1. Clean & Parse Lists ---
    # Function to safe eval string lists like '[a, b, c]'
    def parse_list(x):
        if isinstance(x, list): return x
        try:
            return ast.literal_eval(x)
        except:
            return []

    print("Parsing lists in queries...")
    df_queries['cleaned_result_base_products'] = df_queries['result_base_products'].apply(parse_list)
    df_queries['cleaned_viewed_base_products'] = df_queries['viewed_base_products'].apply(parse_list)
    
    # --- 2. Create Mappings ---
    # Product Map: UID -> {text, category_id}
    print("Creating product mappings...")
    product_map = {}
    
    # We need to map category_ids to 0..N for CrossEntropy
    # Collect all unique categories from products
    all_cats = df_products['category_id'].unique()
    cat_encoder = LabelEncoder()
    cat_encoder.fit(all_cats)
    NUM_CLASSES = len(all_cats)
    
    for _, row in df_products.iterrows():
        # Cleaned text column name based on image (either 'cleaned_text' or constructed)
        # Using 'cleaned_text' based on context, fallback to persian_name if needed
        text_col = 'cleaned_text' if 'cleaned_text' in df_products.columns else 'persian_name'
        
        # Handle unknown categories in products safely
        cat_idx = cat_encoder.transform([row['category_id']])[0]
        
        product_map[row['uid']] = {
            'text': row[text_col],
            'cat_id': row['category_id'],
            'cat_idx': cat_idx  # Encoded for model
        }
        
    # Group products by category for Type 2/3 negative sampling
    products_by_cat = df_products.groupby('category_id')['uid'].apply(list).to_dict()
    all_product_uids = df_products['uid'].tolist()

    # --- 3. Split Queries ---
    print("Splitting queries...")
    # Filter valid queries (must have at least one click)
    df_queries = df_queries[df_queries['cleaned_viewed_base_products'].apply(len) > 0].copy()
    
    # Encode query categories (handle 0 explicitly or treat as class 0? 
    # Logic: We will mask 0 later, but for encoding, we map valid ones)
    # We assume cat 0 is NOT in the product list, so we map it to -1 or handle carefully
    def get_query_cat_idx(c_id):
        if c_id == 0: return -1 # Special token
        try:
            return cat_encoder.transform([c_id])[0]
        except:
            return -1

    df_queries['cat_idx'] = df_queries['category_id'].apply(get_query_cat_idx)

    # Stratify split if possible, else random
    try:
        train_df, test_df = train_test_split(df_queries, test_size=Config.TEST_SIZE, stratify=df_queries['category_id'], random_state=Config.SEED)
        train_df, val_df = train_test_split(train_df, test_size=Config.VAL_SIZE, stratify=train_df['category_id'], random_state=Config.SEED)
    except ValueError:
        print("Warning: Stratified split failed (probably rare classes). Doing random split.")
        train_df, test_df = train_test_split(df_queries, test_size=Config.TEST_SIZE, random_state=Config.SEED)
        train_df, val_df = train_test_split(train_df, test_size=Config.VAL_SIZE, random_state=Config.SEED)
        
    return {
        "train": train_df, "val": val_df, "test": test_df,
        "product_map": product_map,
        "products_by_cat": products_by_cat,
        "all_product_uids": all_product_uids,
        "num_classes": NUM_CLASSES,
        "cat_encoder": cat_encoder
    }

data_bundle = load_and_prep_data()

In [ ]:
from torch.utils.data import Dataset, DataLoader
import random

class RetrievalDataset(Dataset):
    def __init__(self, df, product_map, products_by_cat, all_product_uids, tokenizer, max_len):
        self.samples = [] # List of triplets (query_text, pos_uid, neg_uid, q_cat_idx)
        self.product_map = product_map
        self.products_by_cat = products_by_cat
        self.all_product_uids = all_product_uids
        self.tokenizer = tokenizer
        self.max_len = max_len
        
        # Pre-generate triplets to ensure stability per epoch (or do it dynamically)
        self._generate_triplets(df)
        
    def _generate_triplets(self, df):
        # Flatten the dataframe: 1 row per CLICK
        for _, row in df.iterrows():
            q_text = row['cleaned_query']
            q_cat_id = row['category_id']
            q_cat_idx = row['cat_idx']
            
            viewed = set(row['cleaned_viewed_base_products'])
            shown = set(row['cleaned_result_base_products'])
            # Negatives Type 1 candidates: Shown but NOT viewed
            shown_not_viewed = list(shown - viewed)
            
            for pos_uid in viewed:
                if pos_uid not in self.product_map: continue
                
                # Determine Negative Type
                rand = random.random()
                neg_uid = None
                
                # --- Type 1: Shown but not clicked ---
                if rand < Config.NEG_TYPE_1_RATIO:
                    if shown_not_viewed:
                        neg_uid = random.choice(shown_not_viewed)
                
                # --- Type 2: Random same category ---
                elif rand < (Config.NEG_TYPE_1_RATIO + Config.NEG_TYPE_2_RATIO):
                    # Get category of the POSITIVE product (usually matches query, but use product's truth)
                    pos_cat = self.product_map[pos_uid]['cat_id']
                    candidates = self.products_by_cat.get(pos_cat, [])
                    # Exclude viewed
                    candidates = [c for c in candidates if c not in viewed]
                    if candidates:
                        neg_uid = random.choice(candidates)

                # --- Type 3: Random other category (Fallthrough if T1/T2 failed) ---
                if neg_uid is None or neg_uid not in self.product_map:
                    # Just pick random product that wasn't viewed
                    while True:
                        cand = random.choice(self.all_product_uids)
                        if cand not in viewed and cand in self.product_map:
                            neg_uid = cand
                            break
                
                self.samples.append({
                    'q_text': q_text,
                    'q_cat_idx': q_cat_idx, # -1 if cat_id is 0
                    'pos_uid': pos_uid,
                    'neg_uid': neg_uid
                })

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        item = self.samples[idx]
        
        # Get texts
        q_text = item['q_text']
        pos_data = self.product_map[item['pos_uid']]
        neg_data = self.product_map[item['neg_uid']]
        
        # Tokenization
        q_enc = self.tokenizer(q_text, truncation=True, padding='max_length', max_length=self.max_len, return_tensors="pt")
        pos_enc = self.tokenizer(pos_data['text'], truncation=True, padding='max_length', max_length=self.max_len, return_tensors="pt")
        neg_enc = self.tokenizer(neg_data['text'], truncation=True, padding='max_length', max_length=self.max_len, return_tensors="pt")
        
        return {
            'q_input_ids': q_enc['input_ids'].squeeze(0),
            'q_attention_mask': q_enc['attention_mask'].squeeze(0),
            'q_cat_idx': torch.tensor(item['q_cat_idx'], dtype=torch.long),
            
            'pos_input_ids': pos_enc['input_ids'].squeeze(0),
            'pos_attention_mask': pos_enc['attention_mask'].squeeze(0),
            'pos_cat_idx': torch.tensor(pos_data['cat_idx'], dtype=torch.long),
            
            'neg_input_ids': neg_enc['input_ids'].squeeze(0),
            'neg_attention_mask': neg_enc['attention_mask'].squeeze(0),
            'neg_cat_idx': torch.tensor(neg_data['cat_idx'], dtype=torch.long),
        }

In [ ]:
import torch.nn as nn
from transformers import AutoModel, AutoConfig

class EncoderWithHeads(nn.Module):
    def __init__(self, base_model_path, num_classes, embedding_dim):
        super().__init__()
        # Load Base Model
        self.bert = AutoModel.from_pretrained(base_model_path)
        hidden_size = self.bert.config.hidden_size
        
        # Contrastive Head (Projection)
        self.contrastive_head = nn.Sequential(
            nn.Linear(hidden_size, hidden_size),
            nn.ReLU(),
            nn.Dropout(Config.DROPOUT),
            nn.Linear(hidden_size, embedding_dim)
        )
        
        # Classification Head
        self.classification_head = nn.Sequential(
            nn.Linear(hidden_size, hidden_size),
            nn.ReLU(),
            nn.Dropout(Config.DROPOUT),
            nn.Linear(hidden_size, num_classes)
        )
        
    def forward(self, input_ids, attention_mask):
        # Base Model Forward
        outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        # Pooling: Use CLS token (index 0)
        cls_token = outputs.last_hidden_state[:, 0, :]
        
        # Heads
        emb = self.contrastive_head(cls_token)
        # Normalize embedding for Cosine Similarity
        emb = nn.functional.normalize(emb, p=2, dim=1)
        
        logits = self.classification_head(cls_token)
        
        return emb, logits

class RetrievalSystem(nn.Module):
    def __init__(self, base_model_path, num_classes):
        super().__init__()
        # Two DISTINCT encoders, initialized from same weights
        self.query_encoder = EncoderWithHeads(base_model_path, num_classes, Config.PROJECTION_DIM)
        self.product_encoder = EncoderWithHeads(base_model_path, num_classes, Config.PROJECTION_DIM)
        
    def forward(self, batch):
        # Query Forward
        q_emb, q_logits = self.query_encoder(
            batch['q_input_ids'], batch['q_attention_mask']
        )
        
        # Positive Product Forward
        pos_emb, pos_logits = self.product_encoder(
            batch['pos_input_ids'], batch['pos_attention_mask']
        )
        
        # Negative Product Forward
        neg_emb, neg_logits = self.product_encoder(
            batch['neg_input_ids'], batch['neg_attention_mask']
        )
        
        return q_emb, q_logits, pos_emb, pos_logits, neg_emb, neg_logits

In [ ]:
from transformers import AutoTokenizer
import torch.nn.functional as F
import matplotlib.pyplot as plt

# Initialize Tokenizer
tokenizer = AutoTokenizer.from_pretrained(Config.PRETRAINED_MODEL_PATH)

def calculate_loss(model, batch, criterion_cls, device):
    """Shared loss calculation for Train and Val"""
    # Forward Pass
    q_emb, q_logits, pos_emb, pos_logits, neg_emb, neg_logits = model(batch)
    
    # --- 1. Contrastive Loss (InfoNCE) ---
    # Similarity of Query with ALL Positives and Negatives in batch
    sim_q_all_pos = torch.matmul(q_emb, pos_emb.T) / Config.TEMPERATURE
    sim_q_all_neg = torch.matmul(q_emb, neg_emb.T) / Config.TEMPERATURE
    
    # Concatenate: [sim_pos_batch, sim_neg_batch]
    logits_cont = torch.cat([sim_q_all_pos, sim_q_all_neg], dim=1)
    labels_cont = torch.arange(logits_cont.size(0)).to(device)
    
    loss_cont = F.cross_entropy(logits_cont, labels_cont)
    
    # --- 2. Classification Loss ---
    loss_cls_pos = criterion_cls(pos_logits, batch['pos_cat_idx'])
    loss_cls_neg = criterion_cls(neg_logits, batch['neg_cat_idx'])
    
    # Query Classification (Mask where cat_idx == -1)
    q_targets = batch['q_cat_idx']
    q_mask = (q_targets != -1).float()
    
    loss_cls_q_raw = F.cross_entropy(q_logits, torch.where(q_targets==-1, 0, q_targets), reduction='none')
    loss_cls_q = (loss_cls_q_raw * q_mask).sum() / (q_mask.sum() + 1e-8)
    
    loss_cls_total = loss_cls_pos + loss_cls_neg + loss_cls_q
    
    # Total Loss
    loss = loss_cont + (Config.LAMBDA_CLS * loss_cls_total)
    
    return loss, loss_cont, loss_cls_total

def train_epoch(model, dataloader, optimizer, criterion_cls, device, step_losses):
    model.train()
    total_loss = 0
    
    for step, batch in enumerate(tqdm(dataloader, desc="Training")):
        batch = {k: v.to(device) for k, v in batch.items()}
        
        optimizer.zero_grad()
        loss, _, _ = calculate_loss(model, batch, criterion_cls, device)
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
        
        # Record granular loss
        if step % Config.LOG_STEPS == 0:
            step_losses.append(loss.item())
            
    return total_loss / len(dataloader)

def validate(model, dataloader, criterion_cls, device):
    model.eval()
    total_loss = 0
    total_cont = 0
    total_cls = 0
    
    with torch.no_grad():
        for batch in tqdm(dataloader, desc="Validating"):
            batch = {k: v.to(device) for k, v in batch.items()}
            loss, l_cont, l_cls = calculate_loss(model, batch, criterion_cls, device)
            
            total_loss += loss.item()
            total_cont += l_cont.item()
            total_cls += l_cls.item()
            
    avg_loss = total_loss / len(dataloader)
    return avg_loss, total_cont / len(dataloader), total_cls / len(dataloader)

def plot_loss_history(train_step_losses, val_epoch_losses, steps_per_epoch):
    plt.figure(figsize=(12, 5))
    
    # Plot 1: Granular Training Loss
    plt.subplot(1, 2, 1)
    plt.plot(train_step_losses, label='Training Loss (Step)')
    plt.xlabel(f'Steps (x{Config.LOG_STEPS})')
    plt.ylabel('Loss')
    plt.title('Training Loss Trend')
    plt.grid(True, alpha=0.3)
    plt.legend()
    
    # Plot 2: Validation Loss per Epoch
    plt.subplot(1, 2, 2)
    epochs = range(1, len(val_epoch_losses) + 1)
    plt.plot(epochs, val_epoch_losses, marker='o', color='orange', label='Validation Loss')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.title('Validation Loss per Epoch')
    plt.grid(True, alpha=0.3)
    plt.legend()
    
    plt.tight_layout()
    plt.show()

In [ ]:
import faiss
from sklearn.metrics import classification_report

def evaluate_retrieval(model, test_df, product_map, tokenizer, device, k_values=[1, 5, 10]):
    model.eval()
    
    # --- 1. Index All Products ---
    print("Encoding all products for index...")
    all_uids = list(product_map.keys())
    product_embeddings = []
    
    # Process in batches
    bs = Config.BATCH_SIZE
    with torch.no_grad():
        for i in tqdm(range(0, len(all_uids), bs)):
            batch_uids = all_uids[i:i+bs]
            texts = [product_map[u]['text'] for u in batch_uids]
            
            inputs = tokenizer(texts, padding=True, truncation=True, max_length=Config.MAX_LENGTH, return_tensors="pt").to(device)
            # Use PRODUCT encoder
            emb, _ = model.product_encoder(inputs['input_ids'], inputs['attention_mask'])
            product_embeddings.append(emb.cpu().numpy())
            
    product_embeddings = np.vstack(product_embeddings)
    
    # Build FAISS Index (IP = Inner Product = Cosine Sim since vectors are normalized)
    index = faiss.IndexFlatIP(Config.PROJECTION_DIM)
    index.add(product_embeddings)
    
    # --- 2. Encode Test Queries ---
    print("Encoding test queries...")
    query_texts = test_df['cleaned_query'].tolist()
    query_embeddings = []
    query_cat_logits = []
    
    with torch.no_grad():
        for i in tqdm(range(0, len(query_texts), bs)):
            batch_texts = query_texts[i:i+bs]
            inputs = tokenizer(batch_texts, padding=True, truncation=True, max_length=Config.MAX_LENGTH, return_tensors="pt").to(device)
            # Use QUERY encoder
            emb, logits = model.query_encoder(inputs['input_ids'], inputs['attention_mask'])
            query_embeddings.append(emb.cpu().numpy())
            query_cat_logits.append(logits.cpu().numpy())
            
    query_embeddings = np.vstack(query_embeddings)
    query_cat_logits = np.vstack(query_cat_logits)
    
    # --- 3. Retrieval Metrics ---
    print("Running Search...")
    D, I = index.search(query_embeddings, k=max(k_values))
    
    recalls = {k: [] for k in k_values}
    mrrs = {k: [] for k in k_values}
    ndcgs = {k: [] for k in k_values}
    precisions = {k: [] for k in k_values}
    
    # Map product index back to UID
    index_to_uid = {i: u for i, u in enumerate(all_uids)}
    
    for idx, row in enumerate(test_df.itertuples()):
        true_ids = set(row.cleaned_viewed_base_products)
        if not true_ids: continue
        
        retrieved_indices = I[idx] # Top K indices
        retrieved_uids = [index_to_uid[i] for i in retrieved_indices]
        
        # Calculate Metrics
        for k in k_values:
            top_k = retrieved_uids[:k]
            hits = [1 if uid in true_ids else 0 for uid in top_k]
            
            # Recall@K
            recalls[k].append(sum(hits) / len(true_ids))
            
            # Precision@K
            precisions[k].append(sum(hits) / k)
            
            # MRR@K
            try:
                first_hit = hits.index(1)
                mrrs[k].append(1.0 / (first_hit + 1))
            except ValueError:
                mrrs[k].append(0.0)
                
            # NDCG@K
            dcg = sum([rel / np.log2(i + 2) for i, rel in enumerate(hits)])
            idcg = sum([1.0 / np.log2(i + 2) for i in range(min(len(true_ids), k))])
            ndcgs[k].append(dcg / idcg if idcg > 0 else 0)

    print("\n--- Retrieval Results ---")
    for k in k_values:
        print(f"K={k}: Recall={np.mean(recalls[k]):.4f}, MRR={np.mean(mrrs[k]):.4f}, NDCG={np.mean(ndcgs[k]):.4f}, Prec={np.mean(precisions[k]):.4f}")

    # --- 4. Classification Metrics ---
    print("\n--- Classification Report (Queries) ---")
    # Filter only valid categories
    true_cats = test_df['cat_idx'].tolist()
    pred_cats = np.argmax(query_cat_logits, axis=1)
    
    valid_indices = [i for i, c in enumerate(true_cats) if c != -1]
    y_true = [true_cats[i] for i in valid_indices]
    y_pred = [pred_cats[i] for i in valid_indices]
    
    print(classification_report(y_true, y_pred, zero_division=0))
    
    return query_embeddings, query_cat_logits

In [ ]:
import plotly.express as px
import pandas as pd
from sklearn.manifold import TSNE

def visualize_interactive_tsne(embeddings, labels, cat_encoder, title="T-SNE"):
    # Subsample if too large for performance
    if len(embeddings) > 3000:
        idx = np.random.choice(len(embeddings), 3000, replace=False)
        embeddings = embeddings[idx]
        labels = np.array(labels)[idx]
    
    # Decode labels to actual category names
    cat_names = cat_encoder.inverse_transform(labels)
    
    print(f"Running T-SNE for {title}...")
    tsne = TSNE(n_components=2, random_state=42, perplexity=30, init='pca', learning_rate='auto')
    emb_2d = tsne.fit_transform(embeddings)
    
    # Create DataFrame for Plotly
    df_plot = pd.DataFrame({
        'x': emb_2d[:, 0],
        'y': emb_2d[:, 1],
        'Category': cat_names
    })
    
    fig = px.scatter(
        df_plot, x='x', y='y', color='Category',
        title=title,
        hover_data=['Category'],
        opacity=0.7
    )
    fig.update_layout(width=900, height=700)
    fig.show()

def get_product_embeddings_for_viz(model, product_map, tokenizer, device, num_samples=3000):
    """Encodes a subset of products for visualization"""
    model.eval()
    all_uids = list(product_map.keys())
    
    # Sample random uids
    if len(all_uids) > num_samples:
        sampled_uids = np.random.choice(all_uids, num_samples, replace=False)
    else:
        sampled_uids = all_uids
        
    embeddings = []
    labels = []
    
    with torch.no_grad():
        for i in range(0, len(sampled_uids), Config.BATCH_SIZE):
            batch_uids = sampled_uids[i:i+Config.BATCH_SIZE]
            texts = [product_map[u]['text'] for u in batch_uids]
            cat_idxs = [product_map[u]['cat_idx'] for u in batch_uids]
            
            inputs = tokenizer(texts, padding=True, truncation=True, max_length=Config.MAX_LENGTH, return_tensors="pt").to(device)
            emb, _ = model.product_encoder(inputs['input_ids'], inputs['attention_mask'])
            
            embeddings.append(emb.cpu().numpy())
            labels.extend(cat_idxs)
            
    return np.vstack(embeddings), np.array(labels)

In [ ]:
def main():
    # 1. Prepare Data
    train_df = data_bundle['train']
    val_df = data_bundle['val']
    test_df = data_bundle['test']
    
    # Train Dataset
    train_ds = RetrievalDataset(train_df, data_bundle['product_map'], 
                                data_bundle['products_by_cat'], data_bundle['all_product_uids'],
                                tokenizer, Config.MAX_LENGTH)
    train_loader = DataLoader(train_ds, batch_size=Config.BATCH_SIZE, shuffle=True, num_workers=0)
    
    # Val Dataset
    val_ds = RetrievalDataset(val_df, data_bundle['product_map'], 
                              data_bundle['products_by_cat'], data_bundle['all_product_uids'],
                              tokenizer, Config.MAX_LENGTH)
    val_loader = DataLoader(val_ds, batch_size=Config.VAL_BATCH_SIZE, shuffle=False, num_workers=0)
    
    # 2. Init Model
    model = RetrievalSystem(Config.PRETRAINED_MODEL_PATH, data_bundle['num_classes']).to(Config.DEVICE)
    
    optimizer = torch.optim.AdamW(model.parameters(), lr=Config.LEARNING_RATE, weight_decay=Config.WEIGHT_DECAY)
    criterion_cls = nn.CrossEntropyLoss()
    
    # 3. Training Loop
    train_step_losses = []
    val_epoch_losses = []
    
    print("Starting Training...")
    for epoch in range(Config.NUM_EPOCHS):
        # Train
        train_loss = train_epoch(model, train_loader, optimizer, criterion_cls, Config.DEVICE, train_step_losses)
        
        # Validate
        val_loss, val_cont, val_cls = validate(model, val_loader, criterion_cls, Config.DEVICE)
        val_epoch_losses.append(val_loss)
        
        print(f"Epoch {epoch+1}/{Config.NUM_EPOCHS} | Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f} (Cont: {val_cont:.4f}, Cls: {val_cls:.4f})")
        
    # 4. Save & Plot
    torch.save(model.state_dict(), "retrieval_system_full.pth")
    print("Model saved.")
    
    plot_loss_history(train_step_losses, val_epoch_losses, len(train_loader))
    
    # 5. Evaluation
    q_embs, q_logits = evaluate_retrieval(model, test_df, data_bundle['product_map'], tokenizer, Config.DEVICE)
    
    # 6. Visualization (Interactive T-SNE)
    print("\n--- Visualizing Embeddings ---")
    cat_encoder = data_bundle['cat_encoder']
    
    # A. Query Embeddings
    test_cats = test_df['cat_idx'].tolist()
    valid_mask = [c != -1 for c in test_cats] # Filter undefined query cats
    visualize_interactive_tsne(q_embs[valid_mask], np.array(test_cats)[valid_mask], cat_encoder, title="Query Embeddings (Test Set)")
    
    # B. Product Embeddings
    prod_embs, prod_labels = get_product_embeddings_for_viz(model, data_bundle['product_map'], tokenizer, Config.DEVICE)
    visualize_interactive_tsne(prod_embs, prod_labels, cat_encoder, title="Product Embeddings (Sample)")

if __name__ == "__main__":
    main()